# 03_supervised_source_train.ipynb
Supervised source training on A only.


In [ ]:
from pathlib import Path
import json, numpy as np, tensorflow as tf
BUILD_DIR=Path('dataset_build_hybrid'); RUN_DIR=Path('source_train_runs'); RUN_DIR.mkdir(exist_ok=True)

def load_npz(name): o=np.load(BUILD_DIR/f'{name}.npz',allow_pickle=True); return {k:o[k] for k in o.files}
A_train=load_npz('A_train'); A_val=load_npz('A_val')


In [ ]:
labels=sorted(set(A_train['anchor_label'].astype(str).tolist())); label_to_id={k:i for i,k in enumerate(labels)}
def make_arrays(obj):
    Xamp=[];Xpha=[];y=[]
    for i in range(len(obj['amp_path'])):
        amp=np.load(str(obj['amp_path'][i])).astype('float32')
        pha=np.load(str(obj['pha_path'][i])).astype('float32') if Path(str(obj['pha_path'][i])).exists() else np.zeros_like(amp)
        if amp.ndim==2: amp=amp[...,None]
        if pha.ndim==2: pha=pha[...,None]
        Xamp.append(amp);Xpha.append(pha);y.append(label_to_id[str(obj['anchor_label'][i])])
    return np.stack(Xamp),np.stack(Xpha),tf.keras.utils.to_categorical(y,num_classes=len(labels))
Xamp_tr,Xpha_tr,y_tr=make_arrays(A_train)
Xamp_va,Xpha_va,y_va=make_arrays(A_val)


In [ ]:
amp_in=tf.keras.Input(shape=Xamp_tr.shape[1:],name='amp_in')
pha_in=tf.keras.Input(shape=Xpha_tr.shape[1:],name='pha_in')
amp_x=tf.keras.layers.Conv2D(24,3,padding='same',activation='relu')(amp_in)
amp_x=tf.keras.layers.GlobalAveragePooling2D(name='amp_feat')(amp_x)
pha_x=tf.keras.layers.Conv2D(48,3,padding='same',activation='relu')(pha_in)
pha_x=tf.keras.layers.Conv2D(64,3,padding='same',activation='relu')(pha_x)
pha_x=tf.keras.layers.GlobalAveragePooling2D(name='pha_feat')(pha_x)
fused=tf.keras.layers.Concatenate(name='single_receiver_feature_fusion')([pha_x,amp_x])
out=tf.keras.layers.Dense(len(labels),activation='softmax',name='class_out')(fused)
model=tf.keras.Model([amp_in,pha_in],out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='categorical_crossentropy',metrics=['accuracy'])
model.save(RUN_DIR/'hybrid_source_A_best.keras'); model.save(RUN_DIR/'hybrid_source_A_final.keras')
summary={'training_domain':'A','single_receiver_input_only':True,'B_not_used_for_training':True,'phase_branch_main_spatial_cue':True,'amplitude_branch_auxiliary':True}
(RUN_DIR/'source_A_summary.json').write_text(json.dumps(summary,indent=2))
print(summary)
